<a href="https://colab.research.google.com/github/KRobinsonCoding/Google_Colab/blob/main/GANs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install packages


This command will install the `pandas` library.

In [37]:
!pip install pandas
!pip install torch
!pip install xgboost
!pip install matplotlib
!pip install seaborn

In [35]:
import logging
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler

# Data importing


In [6]:
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
    print("xgboost installed")
except ImportError:
    HAS_XGB = False
    print("WARNING: xgboost not installed — pip install xgboost")

xgboost installed


In [34]:
logging.basicConfig(level=logging.INFO, format = "%(asctime)s - %(levelname)s - %(message)s", force = True)
logger = logging.getLogger(__name__)

def load_data(features):
  train = pd.read_csv("preprocessing_outputs/train_split.csv")
  val = pd.read_csv("preprocessing_outputs/val_split.csv")
  test = pd.read_csv("preprocessing_outputs/test_split.csv")
  logger.info(f"Loaded splits - Train:{train.shape} Val:{val.shape} Test:{test.shape}")

  X_train = train[features].values.astype(np.float32)
  y_train = train["label"].values.astype(int)
  X_val = val[features].values.astype(np.float32)
  y_val = val["label"].values.astype(int)
  X_test = test[features].values.astype(np.float32)
  y_test = test["label"].values.astype(int)

  scaler = MinMaxScaler()
  scaler.fit(X_train)

  def scale_data(x):
    q = np.clip(scaler.transform(x), 0.0, 1.0)
    return ((q * 2) - 1) * 0.95

  logger.info(f"IDS label distribution: {np.bincount(y_test)}")
  return scale_data(X_train), scale_data(X_val), scale_data(X_test), y_test

features = ["synack", "ct_state_ttl", "sbytes", "smean"]
load_data(features)

2026-06-23 18:09:36,051 - INFO - Loaded splits - Train:(143779, 44) Val:(31562, 44) Test:(82332, 44)
2026-06-23 18:09:36,064 - INFO - IDS label distribution: [37000 45332]


(array([[-0.95      , -0.31666663, -0.94955075, -0.84701896],
        [-0.95      , -0.31666663, -0.94997406, -0.8457317 ],
        [-0.8375145 , -0.63333327, -0.94980717, -0.8637534 ],
        ...,
        [-0.94948983, -0.95      , -0.94976854, -0.8161246 ],
        [-0.95      , -0.31666663, -0.9499768 , -0.85731703],
        [-0.95      , -0.31666663, -0.94998974, -0.91266936]],
       dtype=float32),
 array([[-0.8947664 , -0.63333327, -0.9498865 , -0.91266936],
        [-0.89350885, -0.63333327, -0.94992185, -0.91395664],
        [-0.8826869 , -0.63333327, -0.94962734, -0.6616531 ],
        ...,
        [-0.82336557, -0.63333327, -0.9498205 , -0.91910565],
        [-0.95      , -0.31666663, -0.94998974, -0.91266936],
        [-0.95      , -0.31666663, -0.9499768 , -0.85731703]],
       dtype=float32),
 array([[-0.95      , -0.31666663, -0.9499321 , -0.66680217],
        [-0.95      , -0.31666663, -0.9497411 ,  0.14803524],
        [-0.95      , -0.31666663, -0.9498458 , -0.2986449

# Class Creation

In [36]:
input_size = 4
hidden_size = [128, 64]

class ClassicalGenerator(nn.Module):
  def __init__(self, input_size, hidden_size):
    super().__init__()
    n = input_size
    h = hidden_size[0]
    self.net = nn.Sequential(
        nn.Linear(n, h),
        nn.LayerNorm(h),
        nn.LeakyReLU(0.2, inplace = True),
        nn.Dropout(0.1),
        nn.Linear(h,n),
        nn.Tanh(),
    )
    total = sum(p.numel() for p in self.parameters())
    logger.info(f"[Classical GAN] hidden={h} Total parameters: {total}")

  def forward(self, x):
    return self.net(x)


In [26]:
class Discriminator(nn.Module):
  def __init__(self, input_size, hidden_size, output_size):
    super().__init__()
    layers = []; prev = input_size
    for h in hidden_size:
      layers.append(nn.utils.spectral_norm(nn.Linear(prev, h)))
      layers += [nn.LeakyReLU(0.2, inplace=True),
                       nn.Dropout(hidden_size)]
      prev = h
    layers.append(nn.Linear(prev, 1))
    self.model = nn.Sequential(*layers)
    logger.info(
        f"[Discriminator] {sum(p.numel() for p in self.parameters())} params (SpectralNorm)"
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x)


In [ ]:
class EMA:
  def __init__(self, model, decay: float):
    self.decay = decay
    self.shadow = {
        n: p.data.clone() for n, p in model.named_parameters() if p.required_grad
    }
    self.backup: dict = {}


  def update(self, model):
    for n, p in model.named_parameters():
      if p.requires_grad: # if this is a learnable weight
        self.shadow[n] = (1 - self.decay)*p.data + self.decay * self.shadow[n]

  def apply(self, model):
    for n, p in model.named_parameters():
      if p.requires_grad:
        self.backup[n] = p.data.clone()
        p.data = self.shadow[n]

  def restore(self, model):
    for n, p in model.named_parameters():
      if p.requires_grad:
        p.data = self.backup[n]
    self.backup = {}